# Olist E-Commerce — Data Cleaning & Exploratory Data Analysis

This notebook focuses on understanding, cleaning, and exploring the Olist Brazilian E-Commerce dataset before performing deeper business analysis.

## 1. Setup and Data Loading

The analysis begins by importing the required libraries and loading the Olist datasets into pandas DataFrames.

In [1]:
import pandas as pd
from data_loader import load_data

### Load the Datasets

The dataset loader imports the Olist CSV files and returns them as a dictionary of pandas DataFrames.

In [2]:
data = load_data()

In [3]:
list(data.keys())

['customers',
 'orders',
 'order_items',
 'order_payments',
 'order_reviews',
 'products',
 'sellers',
 'geolocation',
 'category_translation']

## 2. Orders Dataset Exploration

The `orders` dataset contains one row per order and provides information about the order status, purchase time, approval time, delivery dates, and estimated delivery date.

The orders table is the main starting point for the cleaning and exploratory analysis in this notebook.

In [4]:
orders = data["orders"]

### Preview the Dataset

A small preview helps verify that the dataset was loaded correctly and provides an initial view of its structure.

In [5]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### Dataset Structure

The dataset dimensions and column names are checked first to understand its size and available fields.

In [6]:
orders.shape

(99441, 8)

In [7]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

### Data Types

The data types are inspected to identify columns that may require conversion before further analysis.

In [8]:
orders.dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

### Dataset Information

A summary of the dataset is used to review the number of entries, data types, and non-null values in each column.

In [9]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


## 3. Data Quality Checks

Before cleaning the data, the main quality issues are identified by checking for missing values and duplicate records.

### Missing Values

Missing values are checked across all columns to identify fields that may require further investigation or cleaning.

In [10]:
missing_values = orders.isnull().sum()
missing_values[missing_values > 0]

order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

The columns with missing values are isolated below so that each missing field can be investigated individually.

### Duplicate Records

Duplicate records are checked at both the row level and the order ID level to ensure that each order is represented only once.

In [11]:
orders.duplicated().sum()

np.int64(0)

In [12]:
orders["order_id"].duplicated().sum()

np.int64(0)

## 4. Data Cleaning

### Missing Order Approval Dates

The missing `order_approved_at` values are investigated to determine whether they represent a data-quality issue or a valid characteristic of the order status.

In [13]:
orders.isnull().sum()[orders.isnull().sum() > 0]

order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

In [14]:
orders.isnull().sum() / orders.shape[0] * 100

order_id                         0.000000
customer_id                      0.000000
order_status                     0.000000
order_purchase_timestamp         0.000000
order_approved_at                0.160899
order_delivered_carrier_date     1.793023
order_delivered_customer_date    2.981668
order_estimated_delivery_date    0.000000
dtype: float64

### Investigating Missing Order Approval Dates

In [15]:
orders[orders["order_approved_at"].isnull()]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
1130,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaN,NaN,NaN,2018-09-12 00:00:00
1801,ed3efbd3a87bea76c2812c66a0b32219,191984a8ba4cbb2145acb4fe35b69664,canceled,2018-09-20 13:54:16,NaN,NaN,NaN,2018-10-17 00:00:00
1868,df8282afe61008dc26c6c31011474d02,aa797b187b5466bc6925aaaa4bb3bed1,canceled,2017-03-04 12:14:30,NaN,NaN,NaN,2017-04-10 00:00:00
2029,8d4c637f1accf7a88a4555f02741e606,b1dd715db389a2077f43174e7a675d07,canceled,2018-08-29 16:27:49,NaN,NaN,NaN,2018-09-13 00:00:00
2161,7a9d4c7f9b068337875b95465330f2fc,7f71ae48074c0cfec9195f88fcbfac55,canceled,2017-05-01 16:12:39,NaN,NaN,NaN,2017-05-30 00:00:00
...,...,...,...,...,...,...,...,...
97696,5a00b4d35edffc56b825c3646a99ba9d,6a3bdf004ca96338fb5fad1b8d93c2e6,canceled,2017-07-02 15:38:46,NaN,NaN,NaN,2017-07-25 00:00:00
98415,227c804e2a44760671a6a5697ea549e4,62e7477e75e542243ee62a0ba73f410f,canceled,2017-09-28 15:02:56,NaN,NaN,NaN,2017-10-16 00:00:00
98909,e49e7ce1471b4693482d40c2bd3ad196,e4e7ab3f449aeb401f0216f86c2104db,canceled,2018-08-07 11:16:28,NaN,NaN,NaN,2018-08-10 00:00:00
99283,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,canceled,2018-08-31 16:13:44,NaN,NaN,NaN,2018-10-01 00:00:00


In [16]:
orders[orders["order_approved_at"].isnull()]["order_status"]

1130     canceled
1801     canceled
1868     canceled
2029     canceled
2161     canceled
           ...   
97696    canceled
98415    canceled
98909    canceled
99283    canceled
99347    canceled
Name: order_status, Length: 160, dtype: str

All orders with a missing `order_approved_at` value have a `canceled` status. Therefore, these missing dates represent orders that were canceled before the approval stage rather than incorrect or incomplete data.

No values are filled or imputed because there is no valid approval date to infer for these orders.

### Convert Date Columns

The order timeline columns are converted from text to datetime format so they can be used for time-based analysis.

In [17]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

### Verify Date Conversion

The converted columns are checked to confirm that all order date fields now use the appropriate datetime type.

In [18]:
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

### Check Remaining Missing Dates

The date columns are checked again after conversion to confirm that the expected missing values remain unchanged.

In [19]:
orders[date_columns].isnull().sum()

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

## 5. Order Status Analysis

Order statuses are examined to understand the overall distribution of orders and identify the most common stages in the order lifecycle.

### Order Status Summary

The order status distribution is summarized using both the number of orders and their percentage of the total.

In [20]:
status_summary = orders["order_status"].value_counts().to_frame("orders")
status_summary["percentage"] = (
    status_summary["orders"] / status_summary["orders"].sum() * 100
).round(2)

status_summary

,orders,percentage
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


## 6. Order Timeline

The purchase dates are explored to understand the time period covered by the dataset and how order volume changes over time.

### Purchase Date Range

The earliest and latest purchase timestamps define the period covered by the available order data.

In [21]:
orders["order_purchase_timestamp"].agg(["min", "max"])

min   2016-09-04 21:15:19
max   2018-10-17 17:30:18
Name: order_purchase_timestamp, dtype: datetime64[us]

### Orders by Year

Order volume is summarized by year to compare the number of orders across the available period.

In [22]:
orders["order_purchase_timestamp"].dt.year.value_counts().sort_index()

order_purchase_timestamp
2016      329
2017    45101
2018    54011
Name: count, dtype: int64

### Orders by Month

Monthly order volume is calculated using year-month periods so that the same month from different years is not combined.

In [23]:
orders["order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index()

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64

## 7. Dataset Exploration

The Olist dataset is divided into several related tables. The main supporting datasets are explored to understand their structure and the information available for later analysis.

### Order Items

The `order_items` dataset contains product-level information for each order, including prices and freight values.

In [24]:
order_items = data["order_items"]
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


### Order Payments

The `order_payments` dataset contains information about payment methods, payment values, and installment counts.

In [25]:
order_payments = data["order_payments"]
order_payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


### Products

The `products` dataset contains product-level information such as categories, dimensions, and weight.

In [26]:
products = data["products"]
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


### Customers

The `customers` dataset contains customer information, including customer location and identifiers.

In [27]:
customers = data["customers"]
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


## 8. Customer Exploration

Customer location is explored to understand the geographic distribution of customer records across Brazilian states.

### Customers by State

The number of customer records is summarized by state to identify the geographic distribution of customers in the dataset.

In [28]:
customers["customer_state"].value_counts()

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64

## 9. Product Exploration

Product categories and physical characteristics are explored to understand the variety of products represented in the dataset.

### Product Categories

The number of product records in each category is summarized to identify the most and least represented categories.

In [29]:
products["product_category_name"].value_counts()

product_category_name
cama_mesa_banho                  3029
esporte_lazer                    2867
moveis_decoracao                 2657
beleza_saude                     2444
utilidades_domesticas            2335
                                 ... 
fashion_roupa_infanto_juvenil       5
casa_conforto_2                     5
pc_gamer                            3
seguros_e_servicos                  2
cds_dvds_musicais                   1
Name: count, Length: 73, dtype: int64

### Product Dimensions and Weight

Descriptive statistics are used to examine the distribution and range of product weight and physical dimensions.

In [30]:
products[
    [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].describe()

,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32949.000000,32949.000000,32949.000000,32949.000000
mean,2276.472488,30.815078,16.937661,23.196728
std,4282.038731,16.914458,13.637554,12.079047
min,0.000000,7.000000,2.000000,6.000000
25%,300.000000,18.000000,8.000000,15.000000
50%,700.000000,25.000000,13.000000,20.000000
75%,1900.000000,38.000000,21.000000,30.000000
max,40425.000000,105.000000,105.000000,118.000000


## 10. Dataset Relationships

The Olist data is distributed across multiple related tables. Combining selected tables allows us to explore information that is not available in a single dataset.

### Orders and Customers

The `orders` table is joined with `customers` using `customer_id` to add customer location and identifier information to each order.

Because each `customer_id` appears once in the `customers` table, this merge preserves one row per order.

In [31]:
orders_customers = orders.merge(
    customers,
    on="customer_id",
    how="left"
)

orders_customers.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


The resulting `orders_customers` DataFrame retains the order-level grain while adding customer attributes.

### Orders and Order Items

The `orders` table is joined with `order_items` using `order_id` to combine order information with product-level sales information.

Because one order can contain multiple items, this merge changes the level of detail from one row per order to one row per order item.

In [32]:
orders_items = orders.merge(
    order_items,
    on="order_id",
    how="left"
)

orders_items.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72


The resulting `orders_items` DataFrame is therefore used for item-level calculations such as product revenue. Order-level metrics must be calculated by grouping the data by `order_id` first.

## 11. Initial Business Metrics

A small set of basic business metrics is calculated to establish an initial view of sales performance before moving to the more detailed business analysis in the next notebook.

### Revenue Overview

Total product revenue is calculated from the `price` column in `order_items`. Freight charges are excluded from this metric.

In [33]:
orders_items["price"].sum()

np.float64(13591643.7)

### Average Order Value

The average order value is calculated by first summing product prices for each order and then taking the average across orders.

This avoids giving orders with multiple items more weight than single-item orders.

In [34]:
order_values = orders_items.groupby("order_id")["price"].sum()
order_values.mean()

np.float64(136.68048088816482)

## 12. Data Cleaning Summary

The main cleaning steps completed in this notebook were:

- Investigated missing `order_approved_at` values and confirmed that they belong to canceled orders.
- Converted the order date columns from text to datetime format.
- Checked for duplicate rows and duplicate `order_id` values.
- Preserved valid missing dates instead of filling them with inferred values.
- Kept the original table structure and used joins only when additional attributes were needed for exploration.

## 13. Key Findings

The exploratory analysis revealed several important characteristics of the Olist dataset:

- The dataset contains **99,441 orders** with no duplicate rows or duplicate `order_id` values.
- Most orders have a `delivered` status, while the remaining orders are distributed across several other stages of the order lifecycle.
- The available purchase dates range from **September 2016 to October 2018**.
- The customer base is distributed across all Brazilian states, with **São Paulo (SP)** representing the largest share of customer records.
- The product catalog contains **73 product categories**, with substantial differences in category representation.
- Missing order approval dates were associated with canceled orders and were therefore left unchanged.
- Total product revenue and average order value provide an initial view of sales performance and will be explored in greater detail in the business analysis notebook.

## 14. Conclusion

This notebook prepared the Olist dataset for further analysis by checking data quality, converting date fields, exploring order patterns, and examining the main customer and product datasets.

The cleaned and explored data provides a reliable foundation for the detailed business analysis in the next notebook.